# Filter PDW Healthy Controls Older Than 20

This notebook reads an Excel file with two sheets (`mir_info` and `participants` by default), then returns rows where:

- MRI file name contains `pdw` (configurable keyword)
- Subject is a healthy control (configurable accepted labels)
- Subject age is greater than 20 (configurable threshold)

You can change all parameters in the next cell.

In [11]:
from pathlib import Path
import pandas as pd

# =====================
# Parameters
# =====================
EXCEL_PATH = Path("C:/Projects/thesis_project/Data/OODtest/FOMO300K/joined_meta_data.xlsx")

# Sheet names in the Excel file
MRI_INFO_SHEET = "mri_info"      # change to "mri_info" if needed
PARTICIPANTS_SHEET = "participants"

# Optional explicit column names (set to None for auto-detection)
MRI_SUBJECT_COL = None
MRI_FILE_COL = None
PARTICIPANT_SUBJECT_COL = None
PARTICIPANT_AGE_COL = None
PARTICIPANT_GROUP_COL = None

# Filtering parameters
MODALITY_KEYWORD = "pdw"          # file name must contain this text (case-insensitive)
HEALTHY_CONTROL_VALUES = {
    "healthy control",
    "healthy_control",
    "healthy",
    "hc",
    "control",
}
MIN_AGE = 20
INCLUDE_EQUAL_AGE = False          # False => age > MIN_AGE, True => age >= MIN_AGE

# Output (CSV)
OUTPUT_CSV_PATH = None             # Example: Path("filtered_pdw_hc_over20.csv")

In [12]:
# =====================
# Helpers
# =====================
def normalize_text(value):
    if pd.isna(value):
        return ""
    return str(value).strip().lower()


def resolve_column(df, explicit_name, candidates, df_name):
    if explicit_name:
        if explicit_name not in df.columns:
            raise KeyError(
                f"Column '{explicit_name}' not found in {df_name}. Available columns: {list(df.columns)}"
            )
        return explicit_name

    lower_map = {str(c).strip().lower(): c for c in df.columns}
    for candidate in candidates:
        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]

    raise KeyError(
        f"Could not auto-detect a column in {df_name}. "
        f"Tried candidates: {candidates}. Available columns: {list(df.columns)}"
    )

In [13]:
# =====================
# Load data
# =====================
if not EXCEL_PATH.exists():
    raise FileNotFoundError(f"Excel file not found: {EXCEL_PATH}")

xls = pd.ExcelFile(EXCEL_PATH)
print("Available sheets:", list(xls.sheet_names))

if MRI_INFO_SHEET not in xls.sheet_names:
    raise KeyError(f"Sheet '{MRI_INFO_SHEET}' not found. Available: {list(xls.sheet_names)}")
if PARTICIPANTS_SHEET not in xls.sheet_names:
    raise KeyError(f"Sheet '{PARTICIPANTS_SHEET}' not found. Available: {list(xls.sheet_names)}")

df_mri = pd.read_excel(EXCEL_PATH, sheet_name=MRI_INFO_SHEET)
df_participants = pd.read_excel(EXCEL_PATH, sheet_name=PARTICIPANTS_SHEET)

print(f"Rows in {MRI_INFO_SHEET}: {len(df_mri)}")
print(f"Rows in {PARTICIPANTS_SHEET}: {len(df_participants)}")

# =====================
# Resolve columns
# =====================
mri_subject_col = resolve_column(
    df_mri,
    MRI_SUBJECT_COL,
    candidates=["participant_id", "subject_id", "sub_id", "id", "participant"],
    df_name=MRI_INFO_SHEET,
)
mri_file_col = resolve_column(
    df_mri,
    MRI_FILE_COL,
    candidates=["file_name", "filename", "image_path", "path", "scan_path", "filepath"],
    df_name=MRI_INFO_SHEET,
)

participant_subject_col = resolve_column(
    df_participants,
    PARTICIPANT_SUBJECT_COL,
    candidates=["participant_id", "subject_id", "sub_id", "id", "participant"],
    df_name=PARTICIPANTS_SHEET,
)
participant_age_col = resolve_column(
    df_participants,
    PARTICIPANT_AGE_COL,
    candidates=["age", "age_years", "age_at_scan", "ageyrs"],
    df_name=PARTICIPANTS_SHEET,
)
participant_group_col = resolve_column(
    df_participants,
    PARTICIPANT_GROUP_COL,
    candidates=["group", "diagnosis", "dx", "status", "cohort", "health_status"],
    df_name=PARTICIPANTS_SHEET,
)

print("Resolved columns:")
print("  MRI subject column:", mri_subject_col)
print("  MRI file column:", mri_file_col)
print("  Participant subject column:", participant_subject_col)
print("  Participant age column:", participant_age_col)
print("  Participant group column:", participant_group_col)

# =====================
# Filter MRI rows by modality keyword in file name
# =====================
modality_mask = df_mri[mri_file_col].astype(str).str.contains(
    MODALITY_KEYWORD,
    case=False,
    na=False,
)
df_mri_modality = df_mri.loc[modality_mask].copy()

# =====================
# Filter participants by healthy control + age threshold
# =====================
df_participants_filtered = df_participants.copy()
df_participants_filtered[participant_age_col] = pd.to_numeric(
    df_participants_filtered[participant_age_col],
    errors="coerce",
)

healthy_values_norm = {normalize_text(v) for v in HEALTHY_CONTROL_VALUES}
group_norm = df_participants_filtered[participant_group_col].apply(normalize_text)
healthy_mask = group_norm.isin(healthy_values_norm)

if INCLUDE_EQUAL_AGE:
    age_mask = df_participants_filtered[participant_age_col] >= MIN_AGE
else:
    age_mask = df_participants_filtered[participant_age_col] > MIN_AGE

df_participants_filtered = df_participants_filtered.loc[
    healthy_mask & age_mask
].copy()

# =====================
# Merge filtered MRI and filtered participants
# =====================
result = df_mri_modality.merge(
    df_participants_filtered,
    left_on=mri_subject_col,
    right_on=participant_subject_col,
    how="inner",
    suffixes=("_mri", "_participants"),
)

# Add explicit modality label based on filter keyword
result["modality"] = MODALITY_KEYWORD.lower()

# Optional cleanup: deduplicate same subject/file pair if present
subset_cols = [mri_subject_col, mri_file_col]
subset_cols = [c for c in subset_cols if c in result.columns]
if subset_cols:
    result = result.drop_duplicates(subset=subset_cols)

# =====================
# Report
# =====================
print("\nFilter summary")
print("-" * 40)
print(f"MRI rows total: {len(df_mri)}")
print(f"MRI rows with modality '{MODALITY_KEYWORD}': {len(df_mri_modality)}")
print(f"Participants total: {len(df_participants)}")
print(f"Participants healthy and age-filtered: {len(df_participants_filtered)}")
print(f"Final merged rows: {len(result)}")

display(result.head(20))

if OUTPUT_CSV_PATH is not None:
    OUTPUT_CSV_PATH = Path(OUTPUT_CSV_PATH)
    OUTPUT_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
    result.to_csv(OUTPUT_CSV_PATH, index=False)
    print(f"Saved: {OUTPUT_CSV_PATH}")

Available sheets: ['participants', 'mri_info']
Rows in mri_info: 318877
Rows in participants: 42140
Resolved columns:
  MRI subject column: participant_id
  MRI file column: filename
  Participant subject column: participant_id
  Participant age column: age
  Participant group column: group

Filter summary
----------------------------------------
MRI rows total: 318877
MRI rows with modality 'pdw': 3529
Participants total: 42140
Participants healthy and age-filtered: 22683
Final merged rows: 1873


,dataset,participant_id,session_id_mri,filename,Modality,MagneticFieldStrength,Manufacturer,ManufacturersModelName,SoftwareVersions,MRAcquisitionType,...,RepetitionTime,InversionTime,FlipAngle,,session_id_participants,sex,age,handedness,group,modality
0,PT018_HBN,sub-0005,ses-01,sub-0005/ses-01/anat/sub-0005_ses-01_run-1_PDw...,MR,1.5,Siemens,Avanto,syngo_MR_B17,3D,...,0.030,NaN,15.0,PT016_GSP,ses-01,F,21.0,NaN,Control,pdw
2,PT018_HBN,sub-0005,ses-01,sub-0005/ses-01/anat/sub-0005_ses-01_run-2_PDw...,MR,1.5,Siemens,Avanto,syngo_MR_B17,3D,...,0.030,NaN,15.0,PT016_GSP,ses-01,F,21.0,NaN,Control,pdw
4,PT018_HBN,sub-0022,ses-01,sub-0022/ses-01/anat/sub-0022_ses-01_run-1_PDw...,MR,3,Siemens,Prisma_fit,syngo_MR_E11,3D,...,0.032,NaN,15.0,PT030_OpenNeuro/ds004169,ses-01,M,25.0,R,Control,pdw
5,PT018_HBN,sub-0022,ses-01,sub-0022/ses-01/anat/sub-0022_ses-01_run-2_PDw...,MR,3,Siemens,Prisma_fit,syngo_MR_E11,3D,...,0.032,NaN,15.0,PT030_OpenNeuro/ds004169,ses-01,M,25.0,R,Control,pdw
6,PT018_HBN,sub-0023,ses-01,sub-0023/ses-01/anat/sub-0023_ses-01_run-1_PDw...,MR,3,Siemens,Prisma_fit,syngo_MR_E11,3D,...,0.032,NaN,15.0,PT016_GSP,ses-01,M,21.0,NaN,Control,pdw
7,PT018_HBN,sub-0023,ses-01,sub-0023/ses-01/anat/sub-0023_ses-01_run-2_PDw...,MR,3,Siemens,Prisma_fit,syngo_MR_E11,3D,...,0.032,NaN,15.0,PT016_GSP,ses-01,M,21.0,NaN,Control,pdw
8,PT018_HBN,sub-0024,ses-01,sub-0024/ses-01/anat/sub-0024_ses-01_run-1_PDw...,MR,3,Siemens,Prisma_fit,syngo_MR_E11,3D,...,0.032,NaN,15.0,PT030_OpenNeuro/ds004169,ses-01,F,23.0,R,Control,pdw
9,PT018_HBN,sub-0024,ses-01,sub-0024/ses-01/anat/sub-0024_ses-01_run-2_PDw...,MR,3,Siemens,Prisma_fit,syngo_MR_E11,3D,...,0.032,NaN,15.0,PT030_OpenNeuro/ds004169,ses-01,F,23.0,R,Control,pdw
10,PT018_HBN,sub-0026,ses-01,sub-0026/ses-01/anat/sub-0026_ses-01_run-1_PDw...,MR,3,Siemens,TrioTim,syngo_MR_B19,3D,...,0.031,NaN,15.0,PT016_GSP,ses-01,F,23.0,NaN,Control,pdw
12,PT018_HBN,sub-0026,ses-01,sub-0026/ses-01/anat/sub-0026_ses-01_run-2_PDw...,MR,3,Siemens,TrioTim,syngo_MR_B19,3D,...,0.031,NaN,15.0,PT016_GSP,ses-01,F,23.0,NaN,Control,pdw


In [14]:
# Optional: keep only key columns for export/view
KEY_COLUMNS = [
    mri_subject_col,
    mri_file_col,
    participant_age_col,
    participant_group_col,
    "modality",
]
KEY_COLUMNS = [c for c in KEY_COLUMNS if c in result.columns]

result_key = result[KEY_COLUMNS].copy()
display(result_key.head(20))

,participant_id,filename,age,group,modality
0,sub-0005,sub-0005/ses-01/anat/sub-0005_ses-01_run-1_PDw...,21.0,Control,pdw
2,sub-0005,sub-0005/ses-01/anat/sub-0005_ses-01_run-2_PDw...,21.0,Control,pdw
4,sub-0022,sub-0022/ses-01/anat/sub-0022_ses-01_run-1_PDw...,25.0,Control,pdw
5,sub-0022,sub-0022/ses-01/anat/sub-0022_ses-01_run-2_PDw...,25.0,Control,pdw
6,sub-0023,sub-0023/ses-01/anat/sub-0023_ses-01_run-1_PDw...,21.0,Control,pdw
7,sub-0023,sub-0023/ses-01/anat/sub-0023_ses-01_run-2_PDw...,21.0,Control,pdw
8,sub-0024,sub-0024/ses-01/anat/sub-0024_ses-01_run-1_PDw...,23.0,Control,pdw
9,sub-0024,sub-0024/ses-01/anat/sub-0024_ses-01_run-2_PDw...,23.0,Control,pdw
10,sub-0026,sub-0026/ses-01/anat/sub-0026_ses-01_run-1_PDw...,23.0,Control,pdw
12,sub-0026,sub-0026/ses-01/anat/sub-0026_ses-01_run-2_PDw...,23.0,Control,pdw


In [16]:
# Add dataset name to output + list unique datasets
from pathlib import PurePath


def infer_dataset_from_path(path_value):
    """Best-effort dataset inference from a file path."""
    s = str(path_value).replace("\\", "/")
    parts = [p for p in s.split("/") if p]
    if len(parts) < 2:
        return "unknown"

    skip_exact = {
        "anat", "func", "dwi", "fmap", "pet", "derivatives",
        "images", "image", "img", "mri", "rawdata", "sourcedata"
    }

    for part in reversed(parts[:-1]):
        p = part.lower()
        if p.startswith("sub-") or p.startswith("ses-"):
            continue
        if p in skip_exact:
            continue
        return part

    return PurePath(parts[-2]).name


result_with_dataset = result.copy()

# Prefer existing dataset column if present
candidate_dataset_cols = [
    "dataset_name", "dataset", "study", "source_dataset", "cohort", "project"
]
existing_dataset_col = next((c for c in candidate_dataset_cols if c in result_with_dataset.columns), None)

if existing_dataset_col is not None:
    result_with_dataset["dataset_name"] = result_with_dataset[existing_dataset_col].astype(str)

    # Fill missing/empty values by inferring from MRI file path
    missing_mask = result_with_dataset["dataset_name"].isna() | (
        result_with_dataset["dataset_name"].astype(str).str.strip() == ""
    )
    if missing_mask.any():
        result_with_dataset.loc[missing_mask, "dataset_name"] = (
            result_with_dataset.loc[missing_mask, mri_file_col].apply(infer_dataset_from_path)
        )
else:
    # No dataset column available, infer from file paths
    result_with_dataset["dataset_name"] = result_with_dataset[mri_file_col].apply(infer_dataset_from_path)

# Build list of unique dataset names
dataset_names = sorted(
    result_with_dataset["dataset_name"]
    .astype(str)
    .str.strip()
    .replace("", pd.NA)
    .dropna()
    .unique()
    .tolist()
)

print("\nDatasets found:")
for name in dataset_names:
    print(f"- {name}")

print(f"\nTotal datasets: {len(dataset_names)}")

dataset_counts = (
    result_with_dataset.groupby("dataset_name", dropna=False)
    .size()
    .reset_index(name="n_rows")
    .sort_values("n_rows", ascending=False)
)

print("\nRows per dataset:")
display(dataset_counts)

# Final dataframe you can use/export
result_final = result_with_dataset
print(len(result_final))

display(result_final.head(20))


Datasets found:
- PT018_HBN
- PT021_IXI
- PT030_OpenNeuro/ds001555
- PT030_OpenNeuro/ds001942
- PT030_OpenNeuro/ds003499
- PT030_OpenNeuro/ds003508
- PT030_OpenNeuro/ds003563
- PT030_OpenNeuro/ds003849
- PT030_OpenNeuro/ds004815
- PT030_OpenNeuro/ds005215
- PT030_OpenNeuro/ds005374
- PT030_OpenNeuro/ds005559
- PT035_Yale_Brain_Mets_Longitudinal

Total datasets: 13

Rows per dataset:


,dataset_name,n_rows
0,PT018_HBN,887
1,PT021_IXI,577
4,PT030_OpenNeuro/ds003499,186
12,PT035_Yale_Brain_Mets_Longitudinal,53
5,PT030_OpenNeuro/ds003508,46
7,PT030_OpenNeuro/ds003849,35
6,PT030_OpenNeuro/ds003563,33
3,PT030_OpenNeuro/ds001942,18
2,PT030_OpenNeuro/ds001555,17
8,PT030_OpenNeuro/ds004815,16


1873


,dataset,participant_id,session_id_mri,filename,Modality,MagneticFieldStrength,Manufacturer,ManufacturersModelName,SoftwareVersions,MRAcquisitionType,...,InversionTime,FlipAngle,,session_id_participants,sex,age,handedness,group,modality,dataset_name
0,PT018_HBN,sub-0005,ses-01,sub-0005/ses-01/anat/sub-0005_ses-01_run-1_PDw...,MR,1.5,Siemens,Avanto,syngo_MR_B17,3D,...,NaN,15.0,PT016_GSP,ses-01,F,21.0,NaN,Control,pdw,PT018_HBN
2,PT018_HBN,sub-0005,ses-01,sub-0005/ses-01/anat/sub-0005_ses-01_run-2_PDw...,MR,1.5,Siemens,Avanto,syngo_MR_B17,3D,...,NaN,15.0,PT016_GSP,ses-01,F,21.0,NaN,Control,pdw,PT018_HBN
4,PT018_HBN,sub-0022,ses-01,sub-0022/ses-01/anat/sub-0022_ses-01_run-1_PDw...,MR,3,Siemens,Prisma_fit,syngo_MR_E11,3D,...,NaN,15.0,PT030_OpenNeuro/ds004169,ses-01,M,25.0,R,Control,pdw,PT018_HBN
5,PT018_HBN,sub-0022,ses-01,sub-0022/ses-01/anat/sub-0022_ses-01_run-2_PDw...,MR,3,Siemens,Prisma_fit,syngo_MR_E11,3D,...,NaN,15.0,PT030_OpenNeuro/ds004169,ses-01,M,25.0,R,Control,pdw,PT018_HBN
6,PT018_HBN,sub-0023,ses-01,sub-0023/ses-01/anat/sub-0023_ses-01_run-1_PDw...,MR,3,Siemens,Prisma_fit,syngo_MR_E11,3D,...,NaN,15.0,PT016_GSP,ses-01,M,21.0,NaN,Control,pdw,PT018_HBN
7,PT018_HBN,sub-0023,ses-01,sub-0023/ses-01/anat/sub-0023_ses-01_run-2_PDw...,MR,3,Siemens,Prisma_fit,syngo_MR_E11,3D,...,NaN,15.0,PT016_GSP,ses-01,M,21.0,NaN,Control,pdw,PT018_HBN
8,PT018_HBN,sub-0024,ses-01,sub-0024/ses-01/anat/sub-0024_ses-01_run-1_PDw...,MR,3,Siemens,Prisma_fit,syngo_MR_E11,3D,...,NaN,15.0,PT030_OpenNeuro/ds004169,ses-01,F,23.0,R,Control,pdw,PT018_HBN
9,PT018_HBN,sub-0024,ses-01,sub-0024/ses-01/anat/sub-0024_ses-01_run-2_PDw...,MR,3,Siemens,Prisma_fit,syngo_MR_E11,3D,...,NaN,15.0,PT030_OpenNeuro/ds004169,ses-01,F,23.0,R,Control,pdw,PT018_HBN
10,PT018_HBN,sub-0026,ses-01,sub-0026/ses-01/anat/sub-0026_ses-01_run-1_PDw...,MR,3,Siemens,TrioTim,syngo_MR_B19,3D,...,NaN,15.0,PT016_GSP,ses-01,F,23.0,NaN,Control,pdw,PT018_HBN
12,PT018_HBN,sub-0026,ses-01,sub-0026/ses-01/anat/sub-0026_ses-01_run-2_PDw...,MR,3,Siemens,TrioTim,syngo_MR_B19,3D,...,NaN,15.0,PT016_GSP,ses-01,F,23.0,NaN,Control,pdw,PT018_HBN


In [ ]:
# Age distribution for PDW images in each dataset
import numpy as np
import matplotlib.pyplot as plt

# Use result_final if available, otherwise fall back to result_with_dataset/result
if "result_final" in globals():
    df_plot = result_final.copy()
elif "result_with_dataset" in globals():
    df_plot = result_with_dataset.copy()
else:
    df_plot = result.copy()

# Resolve age column robustly
age_candidates = [
    "age",
    "age_participants",
    "age_mri",
    participant_age_col if "participant_age_col" in globals() else None,
]
age_candidates = [c for c in age_candidates if c is not None]
age_col = next((c for c in age_candidates if c in df_plot.columns), None)

if age_col is None:
    raise KeyError(f"No age column found. Available columns: {list(df_plot.columns)}")

# Resolve dataset column robustly
dataset_col = "dataset_name" if "dataset_name" in df_plot.columns else None
if dataset_col is None:
    for c in ["dataset", "study", "source_dataset", "cohort", "project"]:
        if c in df_plot.columns:
            dataset_col = c
            break
if dataset_col is None:
    raise KeyError("No dataset column found. Please run the dataset_name cell first.")

# Clean
plot_df = df_plot[[dataset_col, age_col]].copy()
plot_df[age_col] = pd.to_numeric(plot_df[age_col], errors="coerce")
plot_df[dataset_col] = plot_df[dataset_col].astype(str).str.strip()
plot_df = plot_df.dropna(subset=[dataset_col, age_col])
plot_df = plot_df[plot_df[dataset_col] != ""]

if plot_df.empty:
    raise ValueError("No rows available for age distribution plot after cleaning.")

# Summary table
age_summary = (
    plot_df.groupby(dataset_col)[age_col]
    .agg(["count", "min", "max", "mean", "median", "std"])
    .sort_values("count", ascending=False)
)
print("Age summary by dataset (PDW rows):")
display(age_summary)

# Boxplot across datasets (compact global comparison)
ordered_datasets = age_summary.index.tolist()
plt.figure(figsize=(max(10, len(ordered_datasets) * 0.7), 5))
box_data = [plot_df.loc[plot_df[dataset_col] == d, age_col].values for d in ordered_datasets]
plt.boxplot(box_data, tick_labels=ordered_datasets, showfliers=False)
plt.xticks(rotation=45, ha="right")
plt.ylabel("Age")
plt.title("PDW age distribution per dataset (boxplot)")
plt.tight_layout()
plt.show()

# Per-dataset histograms
n = len(ordered_datasets)
ncols = min(3, n)
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(5 * ncols, 3.2 * nrows), squeeze=False)
axes_flat = axes.flatten()

for i, dataset_name in enumerate(ordered_datasets):
    ax = axes_flat[i]
    ages = plot_df.loc[plot_df[dataset_col] == dataset_name, age_col]
    ax.hist(ages, bins=15)
    ax.set_title(f"{dataset_name} (n={len(ages)})")
    ax.set_xlabel("Age")
    ax.set_ylabel("Count")

for j in range(i + 1, len(axes_flat)):
    axes_flat[j].axis("off")

fig.suptitle("PDW age histograms by dataset", y=1.02)
plt.tight_layout()
plt.show()